In [1]:
import glob
import pandas as pd
import numpy as np
import pyreadstat

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report, balanced_accuracy_score

from xgboost import XGBClassifier


# -------------------------------------------------
# 0. Load multiple SPSS files
# -------------------------------------------------
files = glob.glob("Data/*.sav")
df_list = []

for f in files:
    df_temp, meta = pyreadstat.read_sav(f)
    df_list.append(df_temp)

df = pd.concat(df_list, ignore_index=True)
print("Combined dataset shape:", df.shape)


# -------------------------------------------------
# 1. Replace missing codes
# -------------------------------------------------
missing_codes = [-1, -9, 99, 999]
df = df.replace(missing_codes, np.nan)


# -------------------------------------------------
# 2. Define target (BINARY)
# -------------------------------------------------
target = "GambOwnConseqDV_PGSI_Problem"

y_raw = df[target]

# Binary conversion: any problem vs none
y = (y_raw >= 1).astype(int)

print("\nBinary class distribution:")
print(y.value_counts())


# -------------------------------------------------
# 3. Aggregate gambling participation features
# -------------------------------------------------
gambling_12m = {
    'lottery': ['GAM_LOT12M'],
    'scratchcard': ['GAM_SCR12M'],
    'casino': ['GAM_CAS12M'],
    'betting': ['GAM_BETSP12M', 'GAM_BETEV12M'],
    'bingo': ['GAM_BING12M'],
    'slots': ['GAM_SLOT12M']
}

gambling_4w = {
    'lottery': ['GAM_LOT4W'],
    'scratchcard': ['GAM_SCR4W'],
    'casino': ['GAM_CAS4W'],
    'betting': ['GAM_BETSP4W', 'GAM_BETEV4W'],
    'bingo': ['GAM_BING4W'],
    'slots': ['GAM_SLOT4W']
}

def aggregate_features(df, feature_dict, prefix):
    agg_df = pd.DataFrame(index=df.index)
    for key, cols in feature_dict.items():
        available_cols = [c for c in cols if c in df.columns]
        if available_cols:
            agg_df[f"{prefix}_{key}_count"] = df[available_cols].fillna(0).sum(axis=1)
        else:
            agg_df[f"{prefix}_{key}_count"] = 0
    return agg_df

X_12m = aggregate_features(df, gambling_12m, '12m')
X_4w  = aggregate_features(df, gambling_4w, '4w')


# -------------------------------------------------
# 4. Intensity features
# -------------------------------------------------
intensity_df = pd.DataFrame(index=df.index)

for key in gambling_12m.keys():
    intensity_df[f"{key}_intensity"] = (
        X_12m[f"12m_{key}_count"] * 1 +
        X_4w[f"4w_{key}_count"] * 3
    )


# -------------------------------------------------
# 5. Other predictors (NO PGSI items)
# -------------------------------------------------
other_features = [
    'GAM_STOP_SELFEXCL',
    'GAM_SET_LIMITS',
    'GAM_BLOCK_SOFT',
    'GAM_BANK_BLOCK',
    'GAM_TAKE_BREAK',
    'AGE_GROUP',
    'SEX',
    'ETH_GROUP5',
    'ANNUAL_INCOME',
    'ACTIVITY_STATUS',
    'COUNTRY',
    'SWEMWBS_RAW',
    'AUDIT_C',
    'SUICIDAL_THOUGHTS'
]

X_other = df[[c for c in other_features if c in df.columns]]

# Combine all predictors
X = pd.concat([X_12m, X_4w, intensity_df, X_other], axis=1)


# -------------------------------------------------
# 6. Remove missing target rows
# -------------------------------------------------
mask = y_raw.notna()
X = X[mask]
y = y[mask]


# -------------------------------------------------
# 7. Train-test split
# -------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# -------------------------------------------------
# 8. Preprocessing
# -------------------------------------------------
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_features),
    ('cat', cat_pipeline, categorical_features)
])


# -------------------------------------------------
# 9. Compute scale_pos_weight
# -------------------------------------------------
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("\nscale_pos_weight:", scale_pos_weight)


# -------------------------------------------------
# 10. XGBoost model
# -------------------------------------------------
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    tree_method='hist'
)


# -------------------------------------------------
# 11. Full pipeline
# -------------------------------------------------
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', model)
])


# -------------------------------------------------
# 12. Train
# -------------------------------------------------
model_pipeline.fit(X_train, y_train)


# -------------------------------------------------
# 13. Evaluate
# -------------------------------------------------
y_proba = model_pipeline.predict_proba(X_test)[:, 1]
y_pred = model_pipeline.predict(X_test)

roc_auc = roc_auc_score(y_test, y_proba)

print("\nROC-AUC:", roc_auc)
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


# -------------------------------------------------
# 14. Cross-validated AUC
# -------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model_pipeline, X, y, cv=cv, scoring='roc_auc')

print("\nCV ROC-AUC mean:", np.mean(cv_scores))

Combined dataset shape: (29456, 670)

Binary class distribution:
GambOwnConseqDV_PGSI_Problem
0    28557
1      899
Name: count, dtype: int64

scale_pos_weight: 31.636995827538247

ROC-AUC: 0.5
Balanced accuracy: 0.5
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      5687
           1       0.00      0.00      0.00       180

    accuracy                           0.97      5867
   macro avg       0.48      0.50      0.49      5867
weighted avg       0.94      0.97      0.95      5867



/home/oem/mlenv311/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/oem/mlenv311/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/oem/mlenv311/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



CV ROC-AUC mean: 0.5
